In [1]:
%pip install torchvision

Note: you may need to restart the kernel to use updated packages.


In [2]:
import torch
import torch.nn as nn
import torch.optim as optim

import torchvision
from torchvision.datasets import CIFAR10

In [4]:
# datasets and dataloaders

from torch.utils.data import DataLoader
from torchvision.transforms import transforms

# images=> scaling(0,1)=> normalize(-1,1)

transform=transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
])


In [7]:
trainset=CIFAR10(root="./data", train=True, download=True, transform=transform)
testset=CIFAR10(root="./data", train=False, download=True, transform=transform)

100%|█████████████████████████████████████████████████████████████████████████████| 170M/170M [1:45:29<00:00, 26.9kB/s]


In [10]:
trainloader=DataLoader(trainset, batch_size=64, shuffle=True)
testloader=DataLoader(testset, batch_size=64)

### Build the CNN

In [27]:
class CNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv_layers = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )

        self.fc_layers = nn.Sequential(
            nn.Linear(4 * 4 * 128, 256),
            nn.ReLU(),
            nn.Linear(256, 10)
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = x.view(x.size(0), -1)
        x = self.fc_layers(x)
        return x

In [28]:
model=CNN()

In [31]:
criterian=nn.CrossEntropyLoss()
optimizer=optim.Adam(model.parameters())

### Training the CNN

In [44]:
training_losses=[]
val_losses=[]

# best_val_loss=float("inf")
epochs=10

for epoch in range(epochs):
    model.train()
    running_loss=0.0

    for images, labels in trainloader:
        optimizer.zero_grad()
    
        output=model(images)    #FP
        loss=criterian(output, labels)
        loss.backward()      #BP
        optimizer.step()
    
        running_loss+=loss.item()
    
    epoch_train_loss=running_loss/len(trainloader)
    training_losses.append(epoch_train_loss)
        
    #validation
    model.eval()

    running_val_loss=0.0

    with torch.no_grad():
        for images, labels in testloader:
    
            output=model(images)    #FP
            loss=criterian(output, labels)
    
            running_val_loss+=loss.item()
    
        epoch_val_loss=running_val_loss/len(testloader)
        val_losses.append(epoch_val_loss)


    print(f"epoch={epoch+1}/{epochs}==> training loss={epoch_train_loss} & val loss={epoch_val_loss}")

epoch=1/10==> training loss=0.08202875181353744 & val loss=1.5485547987898445
epoch=2/10==> training loss=0.08620529506972437 & val loss=1.6303301681378843
epoch=3/10==> training loss=0.07605900190493135 & val loss=1.7405675778723066
epoch=4/10==> training loss=0.07461544744523904 & val loss=1.8001032306510172
epoch=5/10==> training loss=0.06786766063774485 & val loss=1.7845116151366265
epoch=6/10==> training loss=0.06567831938851701 & val loss=1.9898534633551435
epoch=7/10==> training loss=0.06034070166700836 & val loss=1.9768446220713816
epoch=8/10==> training loss=0.07748868089364877 & val loss=1.9222055487572007
epoch=9/10==> training loss=0.05966741955288879 & val loss=2.04610831342685
epoch=10/10==> training loss=0.05829750077279113 & val loss=1.95950363054397


In [37]:
# Evaluate our CNN

correct_labels=0
total_labels=0

model.eval()

for images, labels in testloader:
    outputs=model.forward(images)
    _,predicted= torch.max(outputs,1)

    correct_labels+= (predicted==labels).sum().item()
    total_labels+=labels.size(0)

print(f"accuracy= {correct_labels/total_labels * 100}")
    

accuracy= 74.3


In [1]:
import matplotlib.pyplot as plt
import pandas as pd

loss_df=pd.DataFrame({
    "training_loss": training_losses,
    "val_losses": val_losses
})

plt.plot(loss_df["training_loss"], label="training loss")
plt.plot(loss_df["val_losses"], label="val_losses")

plt.xlabel("Epochs")
plt.ylabel("Losses")

plt.legend()

NameError: name 'training_losses' is not defined